# CamemBERT on gold v3: grade on the human 150, then yearly leakage-free scoring

**Runtime → Change runtime type → T4 GPU.** Every step saves to the shared Drive folder and skips work already there,
so after a kick-out just open this notebook in another account and **Run all** again.

Parallel: give each account a different `YEARS` (e.g. A `2016,2017,2018,2019`, B `2020,2021,2022`, C `2023,2024,2025,2026`).
When all years are cached, any account runs with `YEARS = ""` to assemble the final file.

One-time setup: add every Colab account as a member of the shared drive **Drive** (Manage members, role Content manager).

In [ ]:
DRIVE_DIR = "/content/drive/Shareddrives/Drive/ProjectNLP"   # shared drive "Drive"
YEARS = ""          # "" = all years + assemble; or e.g. "2016,2017,2018,2019" for this account's share
GRADE = True        # train on gold v3 train split, grade on the 150 human rows (once, ~10 min)

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import os, subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
!pip -q install "transformers==4.57.6" sentencepiece
os.chdir(DRIVE_DIR); print(os.listdir("data/curated")[:20])

In [ ]:
# 1. Instrument check: does training on Haiku-expanded gold keep (or raise) human-150 QWK? v2 baseline: 0.618
out = "data/curated/finetune/camembert_3class_evaluation_v3.json"
if GRADE and not os.path.exists(out):
    !python3 preprocessing/camembert_clf.py --gold data/curated/sentiment_gold_v3.csv \
        --split data/curated/sentiment_gold_v3_split.csv --out {out}
else:
    print("skip:", out, "exists" if os.path.exists(out) else "GRADE=False")

In [ ]:
# 2. Yearly expanding refit: each year is fitted only on gold rows dated before it (min 700 rows).
#    Each finished year is written to the cache at once; cached years are skipped on rerun.
G = ("--model camembert --min-train 700 --gold data/curated/sentiment_gold_v3.csv "
     "--split data/curated/sentiment_gold_v3_split.csv --cache-dir data/curated/cache_camembert_v3")
if YEARS:
    !python3 preprocessing/score_corpus.py {G} --years {YEARS}
else:
    !python3 preprocessing/score_corpus.py {G} --output data/curated/04_scored_v3_camembert.parquet \
        --metadata data/curated/04_scored_v3_camembert_metadata.json

In [ ]:
!ls -la data/curated/cache_camembert_v3